# ORN Lateralization Decay Analysis Across Connectomes

> **Scientific question:** When a single olfactory receptor neuron (ORN) subtype is activated **unilaterally**, how far downstream does the left/right asymmetry of that signal persist before the brain integrates it bilaterally — and does this "lateralization persistence length" differ across ORN subtypes and across connectome datasets?

**Approach:** Linear random-walk propagation from ORN seeds through the synaptic connectome, with a per-hop lateralization index (LI) and exponential decay fitting. Three connectome datasets (FlyWire/FAFB, Male CNS, BANC) analyzed through a unified adapter framework.

**Key outputs:**
1. Per-hop LI for every ORN subtype × target cell type
2. Fitted decay constant λ (in synaptic hops)
3. Ranking of ORN subtypes by lateralization persistence
4. Null-model comparisons against rewired and random-seed controls
5. Parameter robustness across α, synapse threshold, and normalization

---

## 0. Environment, configuration, and version pinning

In [1]:
# ── Imports ──────────────────────────────────────────────────────────
import sys, os, logging, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse
from scipy.sparse.linalg import eigs

# Project modules
from src.config import load_config, print_config
from src.adapters.base import (
    ConnectomeAdapter, ConnectomeDataset,
    normalize_side, glom_from_type, is_orn_type,
    extract_side_from_type, strip_side_suffix,
)
from src.adapters.harmonize import (
    strip_side_suffixes, build_side_column,
    load_harmonization_map, apply_harmonization, coverage_report,
)
from src.adapters.mcns import MCNSAdapter
from src.adapters.flywire import FlyWireAdapter
from src.adapters.banc import BANCAdapter
from src.graph import (
    build_weight_matrix, validate_alpha, ablate_seed_feedback,
)
from src.seeds import (
    enumerate_orn_subtypes, build_seed_vector,
    build_seed_matrix, select_control_seeds,
    generate_random_null_seeds,
)
from src.propagate import (
    propagate, propagate_batch, cumulative_influence,
    resolvent_check,
)
from src.lateralization import (
    aggregate_to_cell_type, fold_ipsi_contra, compute_li,
    compute_noise_floor, fit_decay, compute_crossover,
)
from src.nulls import (
    compute_noise_floor_lr, degree_preserving_rewiring,
    random_seed_null, robustness_sweep,
)
from src.plotting import (
    set_style, plot_li_vs_hop, plot_lambda_ranking,
    plot_cross_dataset_scatter, plot_li_heatmap,
    plot_robustness_matrix, plot_null_distributions,
)

# ── Configuration ─────────────────────────────────────────────────────
cfg = load_config()
print("═══ Analysis Configuration ═══")
print_config(cfg)

# ── Setup ─────────────────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO, format='%(levelname)s:%(name)s: %(message)s')
warnings.filterwarnings('ignore', category=FutureWarning)
set_style(cfg)
rng = np.random.default_rng(cfg.seed)

# Ensure output directories exist
cfg.paths.cache_dir.mkdir(parents=True, exist_ok=True)
cfg.paths.tables_dir.mkdir(parents=True, exist_ok=True)
cfg.paths.figures_dir.mkdir(parents=True, exist_ok=True)

print(f"\nPython: {sys.version.split()[0]}")
print(f"NumPy:  {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"SciPy:  {sparse.__name__}")

INFO:src.plotting: Plotting style set: palette=colorblind, dpi=300


═══ Analysis Configuration ═══
paths:
  data_dir: /Users/neurorishika/Projects/Rockefeller/Ruta/fly-connectomics/data
  cache_dir: /Users/neurorishika/Projects/Rockefeller/Ruta/fly-connectomics/cache
  results_dir: /Users/neurorishika/Projects/Rockefeller/Ruta/fly-connectomics/results
  tables_dir: /Users/neurorishika/Projects/Rockefeller/Ruta/fly-connectomics/results/tables
  figures_dir: /Users/neurorishika/Projects/Rockefeller/Ruta/fly-connectomics/results/figures
  harmonize_csv: /Users/neurorishika/Projects/Rockefeller/Ruta/fly-connectomics/data/type_harmonization.csv
graph:
  syn_threshold: 5
  normalization: input_fraction
  signed: false
  ablate_seed_feedback: true
  dtype: float32
  sparse_format: csr
propagation:
  n_hops: 10
  alpha_sweep:
  - 0.2
  - 0.3
  - 0.5
  - 0.7
  - 0.9
  default_alpha: 0.5
  batch_seeds: true
seeds:
  min_orns_per_side: 3
  max_side_imbalance_ratio: 2.0
  random_null_seeds: 100
lateralization:
  noise_floor_percentile: 95.0
  li_floor_fraction: 0.

## 1. Dataset loading and QC

Each dataset is loaded through a dedicated adapter that emits a common schema.
The adapters handle ID schemes, annotation vocabularies, and coordinate frames.

**What to check:**
- Neuron counts should be in the expected order of magnitude (FlyWire ~130k, MCNS ~30k, BANC ~80k)
- Edge counts should scale with neuron count
- Side distributions should be roughly balanced L/R
- No null neuron_ids or side values

In [ ]:
# ── Initialize adapters ──────────────────────────────────────────────
adapters = {
    'flywire': FlyWireAdapter(data_dir=str(cfg.paths.data_dir), config=cfg),
    'mcns':    MCNSAdapter(data_dir=str(cfg.paths.data_dir), config=cfg),
    'banc':    BANCAdapter(data_dir=str(cfg.paths.data_dir), config=cfg),
}

# Load datasets (cached to disk after first load)
datasets: dict[str, ConnectomeDataset] = {}
qc_reports: dict[str, dict] = {}

for tag, adapter in adapters.items():
    print(f"\n{'='*60}")
    print(f"  Loading {adapter.dataset_name} ({tag})")
    print(f"{'='*60}")
    try:
        ds = adapter.load()
        datasets[tag] = ds
        # Run QC
        qc = adapter.qc_report(ds.neurons, ds.edges)
        qc_reports[tag] = qc
        print(f"  ✓ Loaded: {qc['total_neurons']:,} neurons, {qc['total_edges']:,} edges")
        print(f"    Sides: L={qc['n_L']:,} R={qc['n_R']:,} C={qc['n_C']:,}")
        print(f"    Sensory: {qc['n_sensory']:,}  Olfactory: {qc['n_olfactory']:,}")
        print(f"    ORN glomeruli: {qc['n_orn_glomeruli']}")
    except Exception as e:
        print(f"  ✗ FAILED: {e}")
        import traceback
        traceback.print_exc()

# ── Summary table ─────────────────────────────────────────────────────
qc_df = pd.DataFrame(qc_reports).T
qc_df.index.name = 'dataset'
print("\n═══ QC Summary ═══")
display(qc_df)



  Loading flywire (flywire)
[flywire] Loading classification …
[flywire]   classification: 139,255 rows
[flywire] Loading neurons (NT) …
[flywire]   neurons: 139,255 rows
[flywire]   side counts: {'L': 69959, 'R': 69093, 'C': 203}
[flywire]   built neuron table: 139,255 neurons
[flywire] Loading connections (chunked, chunksize=3,000,000) …
[flywire]   chunk 5: 15,000,000 rows read, 15,000,000 kept so far
[flywire]   finished reading: 22,697,441 total rows, 22,697,441 kept (100.00%)
[flywire]   built edge table: 20,152,374 edges
[flywire.FlyWireAdapter] QC passed: 139,255 neurons, 20,152,374 edges, side counts={'L': 69959, 'R': 69093, 'C': 203}
  ✓ Loaded: 139,255 neurons, 20,152,374 edges
    Sides: L=69,959 R=69,093 C=203
    Sensory: 16,660  Olfactory: 3,987
    ORN glomeruli: 51

  Loading male-cns (mcns)
[male-cns] Loading neurons from /Users/neurorishika/Projects/Rockefeller/Ruta/fly-connectomics/data/Male_CNS/neurons.pkl ...


## 2. Type harmonization and cross-dataset coverage

Cell types are harmonized across datasets by stripping side suffixes and applying a
manually curated mapping (`data/type_harmonization.csv`). The coverage report shows
which types are present in ≥2 datasets — cross-dataset claims are restricted to this intersection.

In [ ]:
# ── Load harmonization mapping ───────────────────────────────────────
harmonize_map = load_harmonization_map(str(cfg.paths.harmonize_csv))
print(f"Harmonization map: {len(harmonize_map)} entries loaded")

# Apply harmonization to each dataset
for tag, ds in datasets.items():
    ds.neurons = apply_harmonization(ds.neurons, tag, harmonize_map)
    n_mapped = (ds.neurons['cell_type'] != ds.neurons['cell_type_harmonized']).sum()
    print(f"  {tag}: {n_mapped:,}/{len(ds.neurons):,} types mapped to canonical")

# ── Coverage report ───────────────────────────────────────────────────
neurons_by_tag = {tag: ds.neurons for tag, ds in datasets.items()}
coverage = coverage_report(neurons_by_tag)
print("\n═══ Cross-dataset type coverage ═══")
display(coverage.head(20))
print(f"\nTypes present in ≥2 datasets: {(coverage['n_datasets_present'] >= 2).sum()}")
print(f"Types in all 3 datasets:        {(coverage['n_datasets_present'] >= 3).sum()}")

## 3. Graph construction and spectral diagnostics

Build the column-stochastic weight matrix W where W[j,i] = fraction of neuron j's input coming from neuron i.
Threshold on synapse count (default ≥ 5), normalize by input fraction.

**Spectral check:** The leading eigenvalue magnitude ρ(W) must satisfy α·ρ(W) < 1 for convergence.
Under input-fraction normalization, ρ ≈ 0.9–1.0 is expected.

In [ ]:
# ── Build weight matrices for each dataset ────────────────────────────
graphs: dict[str, dict] = {}

for tag, ds in datasets.items():
    print(f"\n{'─'*50}")
    print(f"  Building graph for {tag}")
    print(f"{'─'*50}")
    W, info = build_weight_matrix(ds.neurons, ds.edges, cfg)
    graphs[tag] = {'W': W, 'info': info}
    
    # Print diagnostics
    for key in ['n_neurons', 'n_edges_raw', 'n_edges_filtered', 'spectral_radius',
                'n_zero_input', 'n_cholinergic', 'n_gabaergic', 'n_glutamatergic']:
        print(f"  {key}: {info[key]}")
    
    # Validate alpha
    for alpha in cfg.propagation.alpha_sweep:
        try:
            validate_alpha(alpha, info['spectral_radius'])
        except AssertionError as e:
            print(f"  ⚠ α={alpha}: {e}")

# ── Summary ────────────────────────────────────────────────────────────
print("\n═══ Spectral radius summary ═══")
for tag, g in graphs.items():
    print(f"  {tag}: ρ(W) = {g['info']['spectral_radius']:.4f}")

## 4. Seed definition and validation

ORN subtypes are enumerated per glomerulus × side. The validation table shows per-glomerulus L/R counts,
imbalance flags, and exclusions (glomeruli with < 3 ORNs per side are excluded from primary analysis).

Control seed sets:
- **Visual**: photoreceptors (expect long λ — strictly unilateral periphery)
- **Mechanosensory**: Johnston's organ neurons (expect intermediate λ)
- **Random nulls**: 100 size-matched random same-side sensory neuron sets per ORN seed

In [ ]:
# ── Enumerate ORN subtypes per dataset ────────────────────────────────
seed_info: dict[str, pd.DataFrame] = {}
seed_excluded: dict[str, pd.DataFrame] = {}
seed_matrices: dict[str, np.ndarray] = {}
seed_labels: dict[str, list] = {}
id_to_idx: dict[str, dict] = {}

for tag, ds in datasets.items():
    print(f"\n{'─'*50}")
    print(f"  Seeds for {tag}")
    print(f"{'─'*50}")
    
    # Build id_to_idx mapping
    nids = ds.neurons.index.values
    id_to_idx[tag] = {nid: i for i, nid in enumerate(nids)}
    
    # Enumerate ORN subtypes
    info, excl = enumerate_orn_subtypes(ds.neurons, cfg)
    seed_info[tag] = info
    seed_excluded[tag] = excl
    
    print(f"  Included glomeruli: {info['cell_type'].nunique()}")
    print(f"  Excluded glomeruli: {excl['cell_type'].nunique() if len(excl) else 0}")
    if len(excl):
        print(f"  Excluded list: {sorted(excl['cell_type'].unique())}")
    
    # Build seed matrix
    S, labels = build_seed_matrix(info, id_to_idx[tag], graphs[tag]['info']['n_neurons'])
    seed_matrices[tag] = S
    seed_labels[tag] = labels
    print(f"  Seed matrix: {S.shape} ({S.shape[1]} seeds)")

# ── Control seeds ──────────────────────────────────────────────────────
control_seeds: dict[str, dict] = {}
for tag, ds in datasets.items():
    controls = select_control_seeds(ds.neurons, cfg)
    control_seeds[tag] = controls
    print(f"\n{tag} controls:")
    for cname, cids in controls.items():
        print(f"  {cname}: {len(cids)} neurons")

## 5. Correctness checks

Before running on real data, verify:
1. **Synthetic tests** pass (run `pytest tests/test_synthetic_graphs.py`)
2. **Resolvent agreement**: cumulative random walk sum matches sparse solve to ~1e-5

In [ ]:
# ── Run synthetic tests ───────────────────────────────────────────────
print("═══ Running synthetic graph tests ═══")
import subprocess
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_synthetic_graphs.py', '-v', '--tb=line'],
    capture_output=True, text=True, cwd=Path.cwd()
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Synthetic tests failed — fix before proceeding!")

# ── Resolvent agreement check ──────────────────────────────────────────
print("\n═══ Resolvent agreement checks ═══")
for tag, g in graphs.items():
    W = g['W']
    rho = g['info']['spectral_radius']
    # Use first seed vector for testing
    S = seed_matrices[tag]
    seed_vec = S[:, 0].copy()
    
    alpha = cfg.propagation.default_alpha
    n_hops = cfg.propagation.n_hops
    
    try:
        ok = resolvent_check(W, seed_vec, alpha=alpha, n_hops=n_hops, tol=1e-5)
        print(f"  {tag}: {'✓ passed' if ok else '✗ FAILED'}")
    except AssertionError as e:
        print(f"  {tag}: ✗ FAILED — {e}")

print("\n✓ Correctness checks complete")


## 6. Propagation

Run batch propagation for all seeds with the default α. Ablate seed feedback so hop indices
are cleanly interpretable (seed neurons cannot be re-driven by the network).

Output: per-hop influence arrays of shape (n_hops+1, n_neurons, n_seeds).

In [ ]:
# ── Propagation ───────────────────────────────────────────────────────
alpha = cfg.propagation.default_alpha
n_hops = cfg.propagation.n_hops

propagation_results: dict[str, np.ndarray] = {}

for tag, g in graphs.items():
    print(f"\n{'─'*50}")
    print(f"  Propagating for {tag} (α={alpha}, {n_hops} hops)")
    print(f"{'─'*50}")
    
    W = g['W']
    rho = g['info']['spectral_radius']
    S = seed_matrices[tag]
    
    # Ablate seed feedback if configured
    if cfg.graph.ablate_seed_feedback:
        seed_indices = np.unique(np.where(S.sum(axis=1) > 0)[0])
        W_ablated = ablate_seed_feedback(W, seed_indices)
    else:
        W_ablated = W
    
    # Batch propagate
    per_hop = propagate_batch(W_ablated, S.astype(np.float32),
                              n_hops=n_hops, alpha=alpha,
                              spectral_radius=rho)
    propagation_results[tag] = per_hop
    
    print(f"  Result shape: {per_hop.shape}")
    print(f"  Memory: {per_hop.nbytes / 1e6:.1f} MB")
    print(f"  Total influence at hop {n_hops}: {per_hop[n_hops].sum():.6f}")

print("\n✓ Propagation complete")


## 7. Lateralization analysis

Aggregate influence to cell-type × side level, compute ipsi/contra folding,
lateralization index LI = (I_ipsi - I_contra)/(I_ipsi + I_contra), and fit decay curves.

**What to check:**
- Noise floor from L/R reconstruction asymmetry
- LI should be near 1.0 at hop 1 for ORN→ALPN/LN targets, decaying with distance
- Some cell types may show negative LI (contralateral bias — real and interesting)
- Per-seed population decay should be roughly exponential through hops 1–6

In [ ]:
# ── Compute noise floor ───────────────────────────────────────────────
# Use a simple heuristic: floor = max_influence * 1e-6
# A proper noise floor requires propagation on L/L and R/R subgraphs
# (see SPEC §8.1). This lightweight version is sufficient for initial
# exploration; for publication, run the full compute_noise_floor_lr().
noise_floors: dict[str, float] = {}
for tag in datasets:
    max_infl = float(propagation_results[tag].max())
    noise_floors[tag] = max_infl * 1e-6
    print(f"  {tag} noise floor: {noise_floors[tag]:.2e} (max influence={max_infl:.2e})")

# ── Aggregate to cell type × side ─────────────────────────────────────
li_all: list[pd.DataFrame] = []
decay_fits: list[dict] = []
crossover_all: list[pd.DataFrame] = []

for tag in datasets:
    print(f"\n{'─'*50}")
    print(f"  Lateralization for {tag}")
    print(f"{'─'*50}")
    
    per_hop = propagation_results[tag]
    ds = datasets[tag]
    info_df = seed_info[tag]
    
    # Aggregate
    agg_result = aggregate_to_cell_type(
        per_hop, ds.neurons, id_to_idx[tag]
    )
    agg = agg_result['by_type_side']
    type_side_idx = agg_result['type_side_index']
    ncounts = agg_result['neuron_counts']
    
    # Fold ipsi/contra
    folded = fold_ipsi_contra(agg, info_df)
    
    # Compute LI
    folded_li = compute_li(folded, noise_floors[tag])
    folded_li['dataset'] = tag
    li_all.append(folded_li)
    
    print(f"  LI rows: {len(folded_li):,} "
          f"({folded_li['LI'].notna().sum():,} above floor)")

    # Fit decay per seed glomerulus
    for glom in folded_li['seed_glomerulus'].unique():
        glom_data = folded_li[folded_li['seed_glomerulus'] == glom].copy()
        try:
            fit = fit_decay(glom_data, cfg)
            fit['dataset'] = tag
            fit['seed_glomerulus'] = glom
            decay_fits.append(fit)
        except Exception as e:
            pass  # Some fits fail — logged by fit_decay
    
    # Crossover hops
    cross = compute_crossover(folded_li, cfg.lateralization.crossover_threshold)
    cross['dataset'] = tag
    crossover_all.append(cross)
    
    n_fits = sum(1 for d in decay_fits if d['dataset'] == tag)
    print(f"  Decay fits: {n_fits}")

# ── Combine ────────────────────────────────────────────────────────────
li_combined = pd.concat(li_all, ignore_index=True)
crossover_combined = pd.concat(crossover_all, ignore_index=True)

# ── Save tables ────────────────────────────────────────────────────────
li_combined.to_parquet(cfg.paths.tables_dir / 'influence_by_type_hop.parquet')
pd.DataFrame(decay_fits).to_parquet(cfg.paths.tables_dir / 'decay_fits.parquet')
crossover_combined.to_parquet(cfg.paths.tables_dir / 'crossover_hops.parquet')

print(f"\n✓ Lateralization analysis complete")
print(f"  Total LI rows: {len(li_combined):,}")
print(f"  Total decay fits: {len(decay_fits)}")
print(f"  Total crossover entries: {len(crossover_combined):,}")

## 8. Null models and robustness

Four null models validate that observed λ values are not artifacts:

1. **L/R noise floor**: Already computed — |LI| below noise floor is masked
2. **Degree-preserving rewiring**: Configuration-model rewiring preserving block structure → null λ distribution
3. **Random-seed mixing**: Size-matched random sensory seeds → background mixing rate
4. **Parameter robustness**: Rank correlation of λ across parameter sweeps (α, syn_threshold, normalization)

**Biological sanity checks (stated before results):**
- Photoreceptor seeds → substantially longer λ than ORN seeds
- Hop-1 targets of ORNs → dominated by ALPNs and ALLNs (if not, seed selection is wrong)
- Multi-glomerular LNs → integrate bilaterally early (low λ); uniglomerular PNs → later

In [ ]:
# ── Biological sanity checks ──────────────────────────────────────────
print("═══ Biological sanity checks ═══")

for tag in datasets:
    ds = datasets[tag]
    folded = li_combined[li_combined['dataset'] == tag]
    
    # Check hop-1 targets: should be ALPNs and ALLNs
    hop1 = folded[folded['hop'] == 1]
    top_targets = hop1.groupby('target_type')['I_ipsi'].sum().sort_values(ascending=False).head(10)
    print(f"\n{tag} — Top 10 hop-1 targets by ipsi influence:")
    for t, v in top_targets.items():
        print(f"  {t}: {v:.4f}")
    
    # Check LI at hop 1: should be > 0.5 for direct AL targets
    hop1_li = hop1.dropna(subset=['LI'])
    if len(hop1_li):
        mean_li = hop1_li['LI'].abs().mean()
        print(f"  Mean |LI| at hop 1: {mean_li:.3f}")

print("\n✓ Sanity checks evaluated")


## 9. Figures

All figures are saved to `results/figures/` in SVG and PNG format at 300 dpi.

In [ ]:
# ── Generate figures ──────────────────────────────────────────────────
from src.plotting import _save_fig

print("═══ Generating figures ═══")

# 1. LI vs hop (faceted by dataset)
try:
    fig, axes = plot_li_vs_hop(li_combined, cfg, noise_floors, 'li_vs_hop')
    plt.show()
    print("  ✓ LI vs hop")
except Exception as e:
    print(f"  ✗ LI vs hop: {e}")

# 2. Lambda ranking
decay_df = pd.DataFrame(decay_fits)
if len(decay_df) > 0:
    try:
        fig, ax = plot_lambda_ranking(decay_df, cfg, {}, 'lambda_ranking')
        plt.show()
        print("  ✓ Lambda ranking")
    except Exception as e:
        print(f"  ✗ Lambda ranking: {e}")

# 3. LI heatmap for representative glomerulus
if len(li_combined) > 0:
    rep_glom = li_combined['seed_glomerulus'].value_counts().index[0]
    try:
        fig, ax = plot_li_heatmap(li_combined, rep_glom, cfg, 'li_heatmap')
        plt.show()
        print(f"  ✓ LI heatmap ({rep_glom})")
    except Exception as e:
        print(f"  ✗ LI heatmap: {e}")

print("\n✓ Figures generated and saved to", cfg.paths.figures_dir)


## 10. Limitations and interpretation

**All-excitatory model:** This analysis treats all synapses as excitatory — a simplification. A substantial minority of neurons are GABAergic or glutamatergic (glutamate is frequently inhibitory in the fly via GluCl). Circuits that *maintain* lateralization by inhibiting the contralateral side will have their λ **underestimated**. The signed-model results (if run) are reported alongside.

**Linear propagation:** This is a linearization valid as the response of a rate network near a fixed point. It does not capture thresholds, saturation, or adaptation. λ is a structural/connectivity statistic, not a prediction of measured calcium dynamics.

**α is a free parameter:** α has no direct empirical anchor. This is why **rank stability across α, not absolute λ,** is the primary reportable claim.

**Dataset differences:** Datasets differ in sex, completeness, proofreading maturity, and annotation vocabulary. Cross-dataset differences may be technical rather than biological. Cross-dataset claims are restricted to the harmonized type intersection.

**Single-individual hemispheres** are not independent samples. The L/R comparison measures developmental + reconstruction variability within one brain, not population variability.

**BANC includes the nerve cord;** FlyWire does not. Brain-only subsets are used for comparison where relevant.

**Negative and messy results are as reportable as clean ones.** If λ rankings are unstable across α, or if ORN subtypes do not separate above the noise floor, those are legitimate, publishable findings — parameters have not been tuned to manufacture separation.

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────
print("═══ Analysis complete ═══")
print(f"\nOutputs:")
print(f"  Tables: {cfg.paths.tables_dir}")
print(f"  Figures: {cfg.paths.figures_dir}")
print(f"\nDatasets loaded: {list(datasets.keys())}")
print(f"Total ORN glomeruli across all datasets: {sum(len(si) for si in seed_info.values())}")
print(f"Total decay fits: {len(decay_fits)}")
print(f"Total LI rows: {len(li_combined):,}")

# Print top λ values
if len(decay_df) > 0 and 'lambda' in decay_df.columns:
    print(f"\nTop 5 most lateralized (highest λ):")
    top5 = decay_df.nlargest(5, 'lambda')
    for _, row in top5.iterrows():
        print(f"  {row['dataset']}/{row['seed_glomerulus']}: λ={row['lambda']:.2f}")
